In [ ]:
# @title
# Gerekli kütüphaneler
!pip install -q imageio gymnasium stable-baselines3

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Rectangle
import random
import gymnasium as gym
from gymnasium import spaces
import torch
import torch.nn as nn
import torch.optim as optim
from collections import deque
import imageio
from tqdm import tqdm

# ============================================
# 5x10 GRID, 9 KONTEYNER, İKİ VİNÇ (VİNÇ-2 TAM SERBEST)
# ============================================
class ContainerYardEnv(gym.Env):
    def __init__(self):
        super().__init__()
        self.rows = 5
        self.cols = 10
        self.max_steps = 1000

        self.action_space = spaces.Discrete(6)
        # State boyutu: 2+2+9+9+9+1+1+3+2+1 = 39
        self.state_size = 2 + 2 + 9 + 9 + 9 + 1 + 1 + 3 + 2 + 1
        self.observation_space = spaces.Box(low=0, high=1, shape=(self.state_size,), dtype=np.float32)

        # Konteyner tanımları
        self.containers_original = {
            1: {'cells': [(4,3), (4,4), (4,5)], 'size': (1, 3), 'name': 'K1', 'pickup': (4,3), 'priority': 1, 'color': 'green'},
            2: {'cells': [(0,4), (1,4), (2,4)], 'size': (3, 1), 'name': 'K2', 'pickup': (0,4), 'priority': 2, 'color': 'yellow'},
            3: {'cells': [(3,4), (3,5)], 'size': (1, 2), 'name': 'K3', 'pickup': (3,4), 'priority': 1, 'color': 'green'},
            4: {'cells': [(0,5), (1,5)], 'size': (2, 1), 'name': 'K4', 'pickup': (0,5), 'priority': 3, 'color': 'red'},
            5: {'cells': [(1,7), (1,8)], 'size': (1, 2), 'name': 'K5', 'pickup': (1,7), 'priority': 1, 'color': 'green'},
            6: {'cells': [(2,6), (3,6)], 'size': (2, 1), 'name': 'K6', 'pickup': (2,6), 'priority': 2, 'color': 'yellow'},
            7: {'cells': [(0,3), (1,3)], 'size': (2, 1), 'name': 'K7', 'pickup': (0,3), 'priority': 2, 'color': 'yellow'},
            8: {'cells': [(2,3), (3,3)], 'size': (2, 1), 'name': 'K8', 'pickup': (2,3), 'priority': 2, 'color': 'yellow'},
            9: {'cells': [(4,8), (4,9)], 'size': (1, 2), 'name': 'K9', 'pickup': (4,8), 'priority': 1, 'color': 'green'}
        }

        self.blocked_cells = [(0,0), (0,1), (0,2), (4,0), (4,1), (4,2)]
        self.crane2_home = (4, 7)          # Vinç-2 bekleme noktası
        self.reset()

    def reset(self, seed=None):
        super().reset(seed=seed)
        self.orders = [5, 4, 3, 6]        # K5, K4, K3, K6

        self.crane1_pos = [1, 0]          # Vinç-1 başlangıç
        self.crane2_pos = [4, 7]          # Vinç-2 bekleme
        self.crane2_state = "idle"        # idle, moving_to_pickup, moving_to_temp, returning_home, restoring
        self.crane2_target_id = None
        self.crane2_temp_pos = None
        self.relocated = []               # taşınan konteynerlerin listesi
        self.obstacle_queue = []          # kaldırılacak engellerin kuyruğu

        # Konteyner durumları
        self.containers = {}
        for cid, data in self.containers_original.items():
            self.containers[cid] = {
                'cells': data['cells'].copy(),
                'original_cells': data['cells'].copy(),
                'size': data['size'],
                'name': data['name'],
                'pickup': data['pickup'],
                'original_pickup': data['pickup'],
                'priority': data['priority'],
                'color': data['color'],
                'picked': False,
                'delivered': False,
                'rotated': False,
                'relocated': False
            }

        self.stacking_grid = [[[] for _ in range(3)] for _ in range(3)]
        self.steps = 0
        self.carrying1 = None
        self.carrying2 = None
        self.done = False
        self.rotation_msg = ""
        self.target_pos = None
        self.next_pickup_target = None
        self.expected_priority = None

        self._update_targets()
        self._prev_distance1 = self._get_distance(self.crane1_pos, self._get_current_target())
        return self._get_obs(), {}

    # ---------- Yardımcı Fonksiyonlar ----------
    def _get_distance(self, pos, target):
        if target is None: return 0.0
        return abs(pos[0] - target[0]) + abs(pos[1] - target[1])

    def _get_current_target(self):
        return self.target_pos if self.carrying1 is not None else self.next_pickup_target

    def _get_obs(self):
        obs = np.zeros(self.state_size, dtype=np.float32)
        obs[0] = (self.crane1_pos[0] - 1) / 2.0
        obs[1] = self.crane1_pos[1] / (self.cols - 1)
        obs[2] = (self.crane2_pos[0] - 1) / 2.0
        obs[3] = self.crane2_pos[1] / (self.cols - 1)

        for i in range(1, 10):
            obs[4 + (i-1)] = float(self.containers[i]['picked'])

        for r in range(3):
            for c in range(3):
                stack = self.stacking_grid[r][c]
                obs[13 + r*3 + c] = stack[-1] / 9.0 if stack else 0.0

        for i in range(1, 10):
            obs[22 + (i-1)] = 1.0 if i in self.orders else 0.0

        obs[31] = float(self.carrying1 is not None)
        obs[32] = float(self.carrying2 is not None)

        if self.target_pos is not None:
            obs[33] = (self.target_pos[0] - 1) / 2.0
            obs[34] = self.target_pos[1] / 2.0
            obs[35] = self.target_pos[2] / 3.0

        if self.next_pickup_target is not None:
            obs[36] = self.next_pickup_target[0] / (self.rows - 1)
            obs[37] = self.next_pickup_target[1] / (self.cols - 1)

        if self.expected_priority is not None:
            obs[38] = self.expected_priority / 3.0

        return obs

    def _is_blocked(self, pos, crane_id):
        r, c = pos
        if r < 0 or r >= self.rows or c < 0 or c >= self.cols:
            return True
        if (r, c) in self.blocked_cells:
            return True
        # Diğer vinç her zaman engel
        other = self.crane2_pos if crane_id == 1 else self.crane1_pos
        if (r, c) == tuple(other):
            return True

        # Vinç-1 sadece yük taşırken konteynerler engel
        if crane_id == 1 and self.carrying1 is not None:
            for cid, data in self.containers.items():
                if not data['picked'] and (r, c) in data['cells']:
                    return True

        # Vinç-2 için konteynerler ASLA engel değil (tam serbest)
        if crane_id == 2:
            return False

        return False

    def _is_in_stacking_area(self, pos):
        r, c = pos
        return 1 <= r <= 3 and 0 <= c <= 2

    def _get_container_at(self, pos):
        for cid, data in self.containers.items():
            if not data['picked'] and not data['delivered'] and pos == data['pickup']:
                return cid
        return None

    def _find_empty_temp_spot(self, size):
        """Geçici depolama alanında (tamamen boş hücreler) soldan sağa, yukarıdan aşağıya tara"""
        rows, cols = size
        for r in range(self.rows - rows + 1):
            for c in range(self.cols - cols + 1):
                if self._is_in_stacking_area((r, c)) or (r, c) in self.blocked_cells or (r, c) == self.crane2_home:
                    continue
                valid = True
                for i in range(rows):
                    for j in range(cols):
                        nr, nc = r+i, c+j
                        if (nr, nc) == self.crane2_home or self._is_in_stacking_area((nr, nc)) or (nr, nc) in self.blocked_cells:
                            valid = False
                            break
                        for cid, data in self.containers.items():
                            if not data['picked'] and (nr, nc) in data['cells']:
                                valid = False
                                break
                        if not valid:
                            break
                    if not valid:
                        break
                if valid:
                    return (r, c)
        return None

    def _get_path_obstacles(self, start, end_2d):
        path = []
        r, c = start
        tr, tc = end_2d
        while c < tc: c += 1; path.append((r, c))
        while c > tc: c -= 1; path.append((r, c))
        while r < tr: r += 1; path.append((r, c))
        while r > tr: r -= 1; path.append((r, c))
        obstacles = []
        for pos in path:
            for cid, data in self.containers.items():
                if not data['picked'] and pos in data['cells'] and cid not in obstacles:
                    obstacles.append(cid)
        return obstacles

    def _find_stacking_position(self, container_id, rotated=False):
        size = self.containers[container_id]['size']
        prio = self.containers[container_id]['priority']
        if rotated: size = (size[1], size[0])
        if size[0] > 3 or size[1] > 3: return None
        for level in range(4):
            for r in range(3 - size[0] + 1):
                for c in range(3 - size[1] + 1):
                    ok = True
                    for i in range(size[0]):
                        for j in range(size[1]):
                            stack = self.stacking_grid[r+i][c+j]
                            if len(stack) > level: ok = False; break
                            if level > 0:
                                if len(stack) <= level-1: ok = False; break
                                below = stack[level-1]
                                if prio <= self.containers[below]['priority']: ok = False; break
                        if not ok: break
                    if ok: return (1+r, c, level), size
        return None

    def _update_targets(self):
        if self.carrying1 is not None:
            pos_info = self._find_stacking_position(self.carrying1, rotated=False)
            rotated = False
            if pos_info is None:
                pos_info = self._find_stacking_position(self.carrying1, rotated=True)
                rotated = True
            self.target_pos = pos_info[0] if pos_info else None
            self.rotated_needed = rotated
        else:
            self.target_pos = None
            self.rotated_needed = False

        remaining = [oid for oid in self.orders if not self.containers[oid]['picked'] and not self.containers[oid]['delivered']]
        if remaining:
            remaining.sort(key=lambda cid: (self.containers[cid]['priority'], self._get_distance(self.crane1_pos, self.containers[cid]['pickup'])))
            nxt = remaining[0]
            self.next_pickup_target = self.containers[nxt]['pickup']
            self.expected_priority = self.containers[nxt]['priority']
        else:
            self.next_pickup_target = None
            self.expected_priority = None

    # ---------- Vinç-2 Gelişmiş Engel Kuyruk Sistemi ----------
    def _crane2_logic(self):
        # --- 1. Geri yükleme modu ---
        if all(self.containers[oid]['delivered'] for oid in self.orders) and len(self.relocated) > 0:
            if self.crane2_state != "restoring":
                self.crane2_state = "restoring"
                self.crane2_target_id = None
                self.carrying2 = None

            if self.carrying2 is None:
                if self.crane2_target_id is None and len(self.relocated) > 0:
                    self.crane2_target_id = self.relocated[0][0]
                cid = self.crane2_target_id
                pickup = self.containers[cid]['pickup']
                if tuple(self.crane2_pos) == pickup:
                    self.containers[cid]['picked'] = True
                    self.carrying2 = cid
                else:
                    self._move_crane2_towards(pickup)
            else:
                cid = self.carrying2
                orig_pickup = self.containers[cid]['original_pickup']
                if tuple(self.crane2_pos) == orig_pickup:
                    self.containers[cid]['cells'] = self.containers[cid]['original_cells'].copy()
                    self.containers[cid]['pickup'] = orig_pickup
                    self.containers[cid]['picked'] = False
                    self.containers[cid]['relocated'] = False
                    self.carrying2 = None
                    self.relocated.pop(0)
                    self.crane2_target_id = None
                    if len(self.relocated) == 0:
                        self.crane2_state = "returning_home"
                else:
                    self._move_crane2_towards(orig_pickup)
            return

        # --- 2. Eve dönüş ---
        if self.crane2_state == "returning_home":
            if tuple(self.crane2_pos) == self.crane2_home:
                self.crane2_state = "idle"
            else:
                self._move_crane2_towards(self.crane2_home)
            return

        # --- 3. Engel kuyruğunu güncelle ---
        if self.carrying1 is not None:
            target = self._get_current_target()
            if target is not None:
                if len(target) == 3:
                    target_2d = target[:2]
                else:
                    target_2d = target
                current_obstacles = self._get_path_obstacles(tuple(self.crane1_pos), target_2d)
                # Kuyruğu güncelle (yeni engelleri ekle, olmayanları çıkar)
                self.obstacle_queue = [obs for obs in current_obstacles if obs not in [r[0] for r in self.relocated] and not self.containers[obs]['picked']]
        else:
            self.obstacle_queue.clear()

        # --- 4. Kuyruktaki engelleri sırayla temizle ---
        if self.crane2_state == "idle":
            if self.obstacle_queue:
                next_obs = self.obstacle_queue[0]
                if self.containers[next_obs]['picked']:
                    self.obstacle_queue.pop(0)
                    return
                self.crane2_target_id = next_obs
                self.crane2_state = "moving_to_pickup"
            else:
                # Engel yok, eve dön
                if tuple(self.crane2_pos) != self.crane2_home:
                    self._move_crane2_towards(self.crane2_home)
                return

        if self.crane2_state == "moving_to_pickup":
            cid = self.crane2_target_id
            pickup = self.containers[cid]['pickup']
            if tuple(self.crane2_pos) == pickup:
                self.containers[cid]['picked'] = True
                self.carrying2 = cid
                spot = self._find_empty_temp_spot(self.containers[cid]['size'])
                if spot:
                    self.crane2_temp_pos = spot
                    self.crane2_state = "moving_to_temp"
                else:
                    # Boş yer yoksa pickup'ı iptal et
                    self.containers[cid]['picked'] = False
                    self.carrying2 = None
                    self.crane2_state = "idle"
                    if cid in self.obstacle_queue:
                        self.obstacle_queue.remove(cid)
            else:
                self._move_crane2_towards(pickup)

        elif self.crane2_state == "moving_to_temp":
            if tuple(self.crane2_pos) == self.crane2_temp_pos:
                cid = self.carrying2
                r, c = self.crane2_temp_pos
                rows, cols = self.containers[cid]['size']
                new_cells = [(r+i, c+j) for i in range(rows) for j in range(cols)]
                self.containers[cid]['cells'] = new_cells
                self.containers[cid]['pickup'] = new_cells[0]
                self.containers[cid]['picked'] = False
                self.containers[cid]['relocated'] = True
                self.relocated.append((cid, self.containers[cid]['original_cells'].copy(), self.containers[cid]['original_pickup']))
                self.carrying2 = None
                if cid in self.obstacle_queue:
                    self.obstacle_queue.remove(cid)
                self.crane2_state = "idle"
            else:
                self._move_crane2_towards(self.crane2_temp_pos)

    def _move_crane2_towards(self, target):
        r, c = self.crane2_pos
        tr, tc = target
        if c < tc: new_pos = [r, c+1]
        elif c > tc: new_pos = [r, c-1]
        elif r < tr: new_pos = [r+1, c]
        elif r > tr: new_pos = [r-1, c]
        else: return
        if not self._is_blocked(new_pos, crane_id=2):
            self.crane2_pos = new_pos

    # ---------- Adım (step) ----------
    def step(self, action):
        self.steps += 1
        reward = -0.1
        self.rotation_msg = ""

        self._crane2_logic()

        # Tüm siparişler teslim edildiyse ve relocated boşsa bitir
        if all(self.containers[oid]['delivered'] for oid in self.orders) and len(self.relocated) == 0:
            self.done = True
            reward += 50.0
            return self._get_obs(), reward, self.done, False, {'rotation': self.rotation_msg}

        # Vinç-1 hareketi
        current_target = self._get_current_target()
        if current_target is not None:
            old_dist = self._prev_distance1
            new_dist = self._get_distance(self.crane1_pos, current_target)
            if new_dist < old_dist:
                reward += 0.3
            elif new_dist > old_dist:
                reward -= 0.3
            self._prev_distance1 = new_dist

        if self.carrying1 is None and self.next_pickup_target is not None:
            if tuple(self.crane1_pos) == self.next_pickup_target and action != 4:
                reward -= 0.5

        blocked = False
        if action == 0:   # Up
            new_pos = [self.crane1_pos[0]-1, self.crane1_pos[1]]
            if not self._is_blocked(new_pos, crane_id=1): self.crane1_pos = new_pos
            else: blocked = True
        elif action == 1: # Down
            new_pos = [self.crane1_pos[0]+1, self.crane1_pos[1]]
            if not self._is_blocked(new_pos, crane_id=1): self.crane1_pos = new_pos
            else: blocked = True
        elif action == 2: # Left
            new_pos = [self.crane1_pos[0], self.crane1_pos[1]-1]
            if not self._is_blocked(new_pos, crane_id=1): self.crane1_pos = new_pos
            else: blocked = True
        elif action == 3: # Right
            new_pos = [self.crane1_pos[0], self.crane1_pos[1]+1]
            if not self._is_blocked(new_pos, crane_id=1): self.crane1_pos = new_pos
            else: blocked = True
        elif action == 4: # Pickup
            if self.carrying1 is None:
                cid = self._get_container_at(tuple(self.crane1_pos))
                if cid is not None and cid in self.orders:
                    if self.containers[cid]['priority'] == self.expected_priority:
                        self.containers[cid]['picked'] = True
                        self.carrying1 = cid
                        self._update_targets()
                        reward += 8.0
                        self._prev_distance1 = self._get_distance(self.crane1_pos, self._get_current_target())
                    else: reward -= 5.0
                else: reward -= 2.0
            else: reward -= 2.0
        elif action == 5: # Drop
            if self.carrying1 is not None:
                if not self._is_in_stacking_area(tuple(self.crane1_pos)):
                    reward -= 4.0; self.rotation_msg = "Sadece istif alaninda birakabilirsiniz!"
                elif self.target_pos is None:
                    reward -= 4.0; self.rotation_msg = "Yer yok!"
                elif (self.crane1_pos[0], self.crane1_pos[1]) != (self.target_pos[0], self.target_pos[1]):
                    reward -= 4.0; self.rotation_msg = f"Hedef: {self.target_pos[:2]}"
                else:
                    reward += 2.0
                    cid = self.carrying1
                    pos_info = self._find_stacking_position(cid, rotated=False)
                    rotated = False
                    if pos_info is None:
                        pos_info = self._find_stacking_position(cid, rotated=True)
                        rotated = True
                    if pos_info and pos_info[0] == self.target_pos:
                        if rotated:
                            self.containers[cid]['rotated'] = True
                            self.rotation_msg = f"Rotasyon: {self.containers[cid]['name']}"
                        (r, c, level), size = pos_info
                        for i in range(size[0]):
                            for j in range(size[1]):
                                stack = self.stacking_grid[r-1+i][c+j]
                                while len(stack) < level: stack.append(None)
                                stack.append(cid)
                        self.containers[cid]['delivered'] = True
                        self.carrying1 = None
                        delivered = sum(1 for o in self.orders if self.containers[o]['delivered'])
                        reward += 10.0 * delivered
                        self._update_targets()
                        self._prev_distance1 = self._get_distance(self.crane1_pos, self._get_current_target())
                        if all(self.containers[o]['delivered'] for o in self.orders):
                            reward += 20.0
                    else: reward -= 4.0; self.rotation_msg = "Rotasyonla bile sigmiyor!"
            else: reward -= 2.0

        if blocked:
            reward -= 0.1

        if self.carrying1 is None and self.next_pickup_target is not None:
            if tuple(self.crane1_pos) == self.next_pickup_target:
                reward += 1.0

        if self.steps >= self.max_steps:
            self.done = True

        return self._get_obs(), reward, self.done, False, {'rotation': self.rotation_msg}

    # ---------- Render ----------
    def render(self, ax=None):
        if ax is None:
            fig, ax = plt.subplots(figsize=(12, 6))
        grid = np.zeros((self.rows, self.cols))
        for cell in self.blocked_cells:
            grid[cell[0], cell[1]] = 1
        for r in range(3):
            for c in range(3):
                stack = self.stacking_grid[r][c]
                if stack:
                    top_id = stack[-1]
                    level = len(stack)-1
                    if level >= 1: grid[1+r][c] = 6
                    else:
                        col = self.containers[top_id]['color']
                        grid[1+r][c] = 3 if col=='green' else (4 if col=='yellow' else 5)
                else: grid[1+r][c] = 2
        for cid, data in self.containers.items():
            if not data['picked'] and not data['delivered']:
                for cell in data['cells']:
                    col = data['color']
                    grid[cell[0]][cell[1]] = 3 if col=='green' else (4 if col=='yellow' else 5)
        colors = ['white', 'gray', 'lightblue', 'green', 'gold', 'red', 'purple']
        cmap = mcolors.ListedColormap(colors)
        ax.imshow(grid, cmap=cmap, vmin=0, vmax=6)
        for i in range(self.rows+1): ax.axhline(i-0.5, color='black', lw=0.5)
        for j in range(self.cols+1): ax.axvline(j-0.5, color='black', lw=0.5)
        for cid, data in self.containers.items():
            if not data['picked'] and not data['delivered']:
                cells = data['cells']
                cr = sum(c[0] for c in cells)/len(cells)
                cc = sum(c[1] for c in cells)/len(cells)
                ax.text(cc, cr, data['name'], ha='center', va='center', fontsize=9, fontweight='bold')
        for r in range(3):
            for c in range(3):
                stack = self.stacking_grid[r][c]
                if stack:
                    top = stack[-1]; lvl = len(stack)-1
                    lbl = f'K{top}' + (f'\nL{lvl}' if lvl>0 else '')
                    ax.text(c, 1+r, lbl, ha='center', va='center', fontsize=8, fontweight='bold',
                            color='white' if lvl>=1 else 'black')
        if self.target_pos is not None:
            tr, tc, tl = self.target_pos
            ax.add_patch(Rectangle((tc-0.5, tr-0.5),1,1, lw=3, edgecolor='yellow', fc='none', ls='--'))
        if self.next_pickup_target is not None and self.carrying1 is None:
            pr, pc = self.next_pickup_target
            ec = 'green' if self.expected_priority==1 else ('yellow' if self.expected_priority==2 else 'red')
            ax.add_patch(Rectangle((pc-0.5, pr-0.5),1,1, lw=3, edgecolor=ec, fc='none', ls='--'))
        ax.plot(self.crane1_pos[1], self.crane1_pos[0], 'r*', markersize=20, markeredgecolor='black', markeredgewidth=2)
        if self.carrying1 is not None:
            ax.text(self.crane1_pos[1], self.crane1_pos[0]-0.4, f'[{self.containers[self.carrying1]["name"]}]',
                    ha='center', va='center', fontsize=8, color='red', fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.8))
        ax.plot(self.crane2_pos[1], self.crane2_pos[0], 'b*', markersize=20, markeredgecolor='black', markeredgewidth=2)
        if self.carrying2 is not None:
            ax.text(self.crane2_pos[1], self.crane2_pos[0]-0.4, f'[{self.containers[self.carrying2]["name"]}]',
                    ha='center', va='center', fontsize=8, color='blue', fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.8))
        ax.plot(self.crane2_home[1], self.crane2_home[0], 's', color='cyan', markersize=10, alpha=0.5)
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_title(f'Siparisler: {",".join([self.containers[o]["name"] for o in self.orders])} | '
                     f'V1: {self.containers[self.carrying1]["name"] if self.carrying1 else "Bos"} | '
                     f'V2: {self.containers[self.carrying2]["name"] if self.carrying2 else "Bos"} ({self.crane2_state})\n'
                     f'{self.rotation_msg}', fontsize=10)
        return ax

# ============================================
# DQN AGENT (state_size=39)
# ============================================
class DQN(nn.Module):
    def __init__(self, state_size, action_size):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(state_size, 256), nn.ReLU(),
            nn.Linear(256, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, action_size)
        )
    def forward(self, x): return self.network(x)

class DQNAgent:
    def __init__(self, state_size, action_size):
        self.state_size = state_size; self.action_size = action_size
        self.memory = deque(maxlen=10000)
        self.gamma = 0.95; self.epsilon = 1.0; self.epsilon_min = 0.01; self.epsilon_decay = 0.999
        self.learning_rate = 0.0005; self.batch_size = 64
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = DQN(state_size, action_size).to(self.device)
        self.target_model = DQN(state_size, action_size).to(self.device)
        self.update_target_model()
        self.optimizer = optim.Adam(self.model.parameters(), lr=self.learning_rate)
        self.criterion = nn.MSELoss()
    def update_target_model(self): self.target_model.load_state_dict(self.model.state_dict())
    def remember(self, *args): self.memory.append(args)
    def act(self, state, training=True):
        if training and np.random.rand() <= self.epsilon: return random.randrange(self.action_size)
        state_t = torch.FloatTensor(state).unsqueeze(0).to(self.device)
        with torch.no_grad(): q = self.model(state_t)
        return q.cpu().numpy()[0].argmax()
    def replay(self):
        if len(self.memory) < self.batch_size: return
        batch = random.sample(self.memory, self.batch_size)
        states = np.array([b[0] for b in batch])
        actions = np.array([b[1] for b in batch])
        rewards = np.array([b[2] for b in batch])
        next_states = np.array([b[3] for b in batch])
        dones = np.array([b[4] for b in batch])
        states = torch.FloatTensor(states).to(self.device)
        actions = torch.LongTensor(actions).to(self.device)
        rewards = torch.FloatTensor(rewards).to(self.device)
        next_states = torch.FloatTensor(next_states).to(self.device)
        dones = torch.FloatTensor(dones).to(self.device)
        curr_q = self.model(states).gather(1, actions.unsqueeze(1))
        with torch.no_grad(): max_next_q = self.target_model(next_states).max(1)[0]
        target_q = rewards + (1-dones) * self.gamma * max_next_q
        loss = self.criterion(curr_q.squeeze(), target_q)
        self.optimizer.zero_grad(); loss.backward(); self.optimizer.step()
        if self.epsilon > self.epsilon_min: self.epsilon *= self.epsilon_decay

# ============================================
# EĞİTİM
# ============================================
print("Ortam oluşturuluyor...")
env = ContainerYardEnv()
agent = DQNAgent(env.state_size, 6)
episodes = 1500
print(f"Eğitim başlıyor... ({episodes} bölüm)")
rewards_history = []
for e in tqdm(range(episodes), desc="Eğitim"):
    state, _ = env.reset()
    total_reward = 0
    while True:
        action = agent.act(state)
        next_state, reward, done, _, _ = env.step(action)
        agent.remember(state, action, reward, next_state, done)
        state = next_state
        total_reward += reward
        if done:
            agent.update_target_model()
            break
        if len(agent.memory) > agent.batch_size: agent.replay()
    rewards_history.append(total_reward)
    if (e+1) % 200 == 0:
        avg = np.mean(rewards_history[-200:])
        tqdm.write(f"Bölüm: {e+1}/{episodes}, Ort. Ödül: {avg:.2f}, Epsilon: {agent.epsilon:.3f}")
print("Eğitim tamamlandı!")
plt.figure(figsize=(10,4))
plt.plot(rewards_history); plt.xlabel('Bölüm'); plt.ylabel('Ödül'); plt.title('Eğitim Süreci'); plt.grid(); plt.show()

# ============================================
# TEST VE GIF
# ============================================
print("Test başlıyor...")
test_env = ContainerYardEnv()
state, _ = test_env.reset()
print(f"Siparişler: {', '.join([test_env.containers[o]['name'] for o in test_env.orders])}")
frames = []
fig, ax = plt.subplots(figsize=(12,6))
step = 0
max_steps = 800
while step < max_steps:
    ax.clear()
    test_env.render(ax)
    fig.canvas.draw()
    img = np.array(fig.canvas.renderer.buffer_rgba())
    frames.append(img[:,:,:3])
    action = agent.act(state, training=False)
    state, reward, done, _, _ = test_env.step(action)
    step += 1
    if done:
        ax.clear(); test_env.render(ax); fig.canvas.draw()
        img = np.array(fig.canvas.renderer.buffer_rgba()); frames.append(img[:,:,:3])
        break
plt.close()
gif_path = 'dual_crane_free_crane2.gif'
imageio.mimsave(gif_path, frames, fps=5, loop=0)
print(f"Test tamamlandı ({step} adım), GIF: {gif_path}")
from IPython.display import Image as IPImage
IPImage(filename=gif_path)